# HCCL 简介与集合通信基础

本章是 HCCL 系列课程的入门章节。要理解「集合通信」，需要先回答两个层层递进的问题：

- **为什么** 集群训练离不开集合通信？
- **HCCL** 这个集合通信库到底是什么、由哪些概念组成？

本章共分三节，建议按顺序学习。

## 本章目录

| 小节 | 主题 | 打开 |
| --- | --- | --- |
| **1.1 本章介绍** | 学习目标、全章目录、术语与缩略语 | 本文件 |
| **1.2 集群通信介绍** | 单卡为什么不够用 → 集合通信 → 四类并行方式 → MoE 与 All2All | [1.2 集群通信介绍](./01.02_cluster_comm_intro.ipynb) |
| **1.3 HCCL** | HCCL 是什么、开放生态、四个基础概念与八个常见算子 | [1.3 HCCL](./01.03_hccl_intro.ipynb) |

---

本章课程请从 [集群通信介绍](./01.02_cluster_comm_intro.ipynb) 开始学习。

### 1.2 集群通信介绍（先回答「为什么」）

先讲清楚**集群训练为什么离不开通信**：

1. **单卡训练的两大瓶颈** —— 性能（训练周期不可接受）与内存（模型装不下）；
2. **集合通信** —— 把多张卡「拧成一台逻辑机器」的粘合剂，以及 AllReduce 的作用；
3. **四类常用并行方式** —— 数据并行 DP / 流水并行 PP / 张量并行 TP / 专家并行 EP
   各自切什么、需要什么通信；
4. **稀疏 MoE 与 All2All** —— 专家并行中最核心、也最需要优化的通信。

### 1.3 HCCL（再回答「用什么」）

知道「为什么需要通信」之后，再看**华为给出的答案**：

1. **HCCL 是什么** —— 定义、定位与两大核心特点；
2. **开放生态：分层解耦** —— HCCL 仓与 HCOMM 仓如何分工；
3. **HCCL 基础概念** —— 通信成员 → 通信域 → 通信算子 → 通信算法；
4. **八个常见集合通信算子** —— Gather / Reduce / Broadcast / Scatter /
   AllReduce / AllGather / ReduceScatter / AlltoAll。

## 学习目标

学完本章，你应当能够：

- 说清 HCCL 的定义、定位与两大核心特点；
- 用「通信成员 → 通信域 → 通信算子 → 通信算法」的层次理解集合通信的组织方式；
- 辨认八种常见集合通信算子，并判断它们属于多对一 / 一对多 / 多对多中的哪一类；
- 说明「单卡训练瓶颈」的两个维度（性能、内存）及其成因；
- 区分四类并行方式，并判断它们各自需要什么样的通信操作；
- 理解「集合通信」与「点对点通信」的本质区别。

## 术语与缩略语

为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

| 概念 | 全称 | 解释说明 |
| --- | --- | --- |
| NPU | Neural Network Processing Unit | 神经网络处理单元，昇腾 AI 计算任务的主要执行设备 |
| AI 节点 | AI Server | 昇腾 AI Server，通常是由 8 卡或 16 卡的昇腾 NPU 设备组成的服务器形态的统称 |
| AI 集群 | AI POD | 由多个节点通过交换机（Switch）互联组成，用于分布式训练或推理的系统 |
| Host | — | 与 Device 相连接的 x86 或 ARM 服务器，利用 Device 提供的计算能力完成业务 |
| Device | — | 集成了计算芯片的硬件设备（如 NPU/GPU），通常通过 PCIe 接口与 Host 侧连接 |
| CANN | Compute Architecture for Neural Networks | 华为昇腾的异构计算架构，向上支撑 AI 框架，向下使能昇腾硬件 |
| HCCL | Huawei Collective Communication Library | 华为集合通信库，CANN 的基础组件之一，提供单机多卡、多机多卡间的集合通信能力 |
| HCOMM | Huawei Communication | 通信基础库，沉淀通信域、基础通信能力，与 HCCL 分层解耦 |
| HIXL | Huawei Xfer Library | 昇腾单边通信库，面向集群场景提供点对点数据传输能力 |
| 通信成员 | Rank | 参与通信的最小逻辑实体，可以是一颗芯片、一个 Die、甚至一个进程，一般称为 Rank |
| 通信域 | Communication Domain | 一组通信成员的组合，描述通信范围；一个集群可建多个通信域，一个成员可加入多个通信域 |
| 通信算子 | Operator | 在通信域内完成通信任务的算子；集合通信指所有成员一起参与的通信操作 |
| 通信算法 | Algorithm | 完成通信算子的实现方案；不同拓扑、数据量下有不同方案，实现不同的流量规划策略 |
| 集合通信 | Collective Communication | 一组通信成员都参与、操作语义由所有成员共同定义的全局通信操作 |
| 点对点通信 | Point-to-Point (P2P) | 仅涉及两个通信实体的通信，如 Send/Recv |
| 并行方式 | Parallelism | 将训练任务切分到多卡上的策略，包括数据/流水/张量/专家并行等 |
| MoE | Mixture-of-Experts | 混合专家模型，通过稀疏激活提升模型参数规模下的计算效率 |
| AllReduce | — | 集合通信操作：对多卡数据做归约（如求和）后，把结果广播回所有卡 |
| All2All | All-to-All | 集合通信操作：每张卡把自己的数据分片发给所有其他卡，是 MoE 专家并行的核心 |